# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [2]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/dahye411/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

/content/2026-HYU-AUE8088-PA2


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)   # 재현성을 위한 시드 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 92.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 22.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Weights & Biases 설정

학습 곡선과 속성별 Macro-F1 을 자동으로 로깅합니다. 사용하지 않으려면 `WANDB_PROJECT = None` 으로 두세요 — 로깅 호출은 자동으로 no-op 이 됩니다.

최초 1회만:
```python
import wandb; wandb.login()   # API key 입력
```

In [5]:
import wandb; wandb.login()   # API key 입력

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sere411 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level1"]
# 환경변수로도 끌 수 있음:  os.environ["WANDB_DISABLED"] = "true"

In [7]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Set A 의 train/val 로더 구성
train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 출력 — 불균형 정도를 직접 확인할 것
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=15d76fb6-dd35-4bd3-9f2a-52cd009a102e
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:01<00:00, 199MB/s] 


압축 해제 중...
완료 → ../data/set_a
weather    클래스별 샘플 수 (train) = [3100, 800, 400, 200, 0, 500]
scene      클래스별 샘플 수 (train) = [3052, 1386, 562]
timeofday  클래스별 샘플 수 (train) = [2479, 2129, 392]


In [8]:
from torch import nn

def train_one(model_fn, name, epochs=20, loss_weights=None):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    loss_weights = loss_weights
    cfg = TrainConfig(epochs=epochs, loss_weights=loss_weights)

    # wandb 로거 — WANDB_PROJECT 가 None 이거나 wandb 미설치 시 자동 no-op
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후 — 속성별 정규화 confusion matrix 를 wandb 에 업로드
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history},
               f"../checkpoints/level1_{name}.pth")

    model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model, history


# 분석 1, 2 용 학습) Skip connetion 유무에 따른 수렴 차이 비교 및 각 백본 별 가장 어려운 속성과 깊이 변화에 따른 성능 변화 분석
vgg_model, vgg_hist = train_one(
    VGG16,
    "vgg16_w111",
    epochs=30,
    loss_weights={"weather": 1.0, "scene": 1.0, "timeofday": 1.0},
)

r18_model, r18_hist = train_one(
    resnet18,
    "resnet18_w111",
    epochs=30,
    loss_weights={"weather": 1.0, "scene": 1.0, "timeofday": 1.0},
)

r50_model, r50_hist = train_one(
    resnet50,
    "resnet50_w111",
    epochs=30,
    loss_weights={"weather": 1.0, "scene": 1.0, "timeofday": 1.0},
)

# 분석 3 용 학습) Loss 가중치 민감도 분석 (ResNet-18 고정, weight만 변경)
r18_w211_model, r18_w211_hist = train_one(
    resnet18,
    "resnet18_w211",
    epochs=30,
    loss_weights={"weather": 2.0, "scene": 1.0, "timeofday": 1.0},
)

r18_w121_model, r18_w121_hist = train_one(
    resnet18,
    "resnet18_w121",
    epochs=30,
    loss_weights={"weather": 1.0, "scene": 2.0, "timeofday": 1.0},
)

r18_w112_model, r18_w112_hist = train_one(
    resnet18,
    "resnet18_w112",
    epochs=30,
    loss_weights={"weather": 1.0, "scene": 1.0, "timeofday": 2.0},
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=4.1137  val_avg_MF1=0.3393  per={'weather': 0.125, 'scene': 0.2531017369727047, 'timeofday': 0.6397102722674687}


[epoch 02/30] train_loss=2.5311  val_avg_MF1=0.3396  per={'weather': 0.125, 'scene': 0.2525879917184265, 'timeofday': 0.6411762801154993}


[epoch 03/30] train_loss=2.3278  val_avg_MF1=0.3760  per={'weather': 0.18671932802367586, 'scene': 0.30156145018678276, 'timeofday': 0.6395997929187206}


[epoch 04/30] train_loss=2.2513  val_avg_MF1=0.4022  per={'weather': 0.19623233908948193, 'scene': 0.3707133058984911, 'timeofday': 0.6397813136943572}


...

[epoch 27/30] train_loss=1.2268  val_avg_MF1=0.6389  per={'weather': 0.5010251232931447, 'scene': 0.6157700303883681, 'timeofday': 0.7998148991787947}


[epoch 28/30] train_loss=1.2032  val_avg_MF1=0.6427  per={'weather': 0.5237389512570191, 'scene': 0.6038797515175468, 'timeofday': 0.8005523444061802}


[epoch 29/30] train_loss=1.1886  val_avg_MF1=0.6458  per={'weather': 0.5285021113071446, 'scene': 0.6006136950904392, 'timeofday': 0.8081476007970881}


[epoch 30/30] train_loss=1.1857  val_avg_MF1=0.6442  per={'weather': 0.526852552691388, 'scene': 0.5952996606420273, 'timeofday': 0.8103592047402408}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
val/avg_macro_f1,▁▂▁▂▂▂▂▃▃▃▄▃▃▆▄▅▆▅▅▅▇▆▇██▇▇███
val/mf1_scene,▂▃▁▂▄▄▁▃▂▂▃▃▂▇▂▅▆▄▄▃▇▇███▇██▇▇
val/mf1_timeofday,▆▆▆▅▁▇▂▆▆▄▆▃▆▄▆▄▆▆▅▆▅▇▇█▆▄▆▆▇▇
val/mf1_weather,▂▂▁▃▃▁▅▄▄▅▅▄▃▆▅▇▆▇▆▆█▆▇███▇███
epoch,30
lr,0
train/loss,1.18569
val/avg_macro_f1,0.64417


## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.

In [ ]:
# 분석 1, 2) 백본별 best epoch 기준 성능 표
# - VGG-16 vs ResNet-18: skip connection 유무 비교
# - ResNet-18 vs ResNet-50: skip connection이 있는 상태에서 깊이 증가 효과 비교
# - hardest_attribute: 세 task 중 best epoch에서 Macro-F1이 가장 낮은 속성

import pandas as pd
import matplotlib.pyplot as plt


def best_summary(name, hist):
    """Return best-epoch Avg-MF1 and per-attribute MF1 from a trainer history."""
    best_epoch = max(
        range(len(hist["val_avg_mf1"])),
        key=lambda i: hist["val_avg_mf1"][i],
    )
    per = hist["val_per_mf1"][best_epoch]
    return {
        "run": name,
        "best_epoch": best_epoch + 1,
        "avg_mf1": hist["val_avg_mf1"][best_epoch],
        "weather_mf1": per["weather"],
        "scene_mf1": per["scene"],
        "timeofday_mf1": per["timeofday"],
    }


def add_hardest_attribute(df):
    attr_cols = ["weather_mf1", "scene_mf1", "timeofday_mf1"]
    out = df.copy()
    out["hardest_attribute"] = out[attr_cols].idxmin(axis=1).str.replace("_mf1", "", regex=False)
    return out

backbone_df = pd.DataFrame([
    best_summary("vgg16_w111", vgg_hist),
    best_summary("resnet18_w111", r18_hist),
    best_summary("resnet50_w111", r50_hist),
])

backbone_df.insert(1, "skip_connection", ["No", "Yes", "Yes"])
backbone_df.insert(2, "depth", [16, 18, 50])
backbone_df = add_hardest_attribute(backbone_df)

backbone_df

In [ ]:
# 분석 1) 수렴 비교

plt.figure(figsize=(8, 5))
plt.plot(vgg_hist["train_loss"], label="VGG-16")
plt.plot(r18_hist["train_loss"], label="ResNet-18")
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title("Convergence: VGG-16 vs ResNet-18")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 분석 2) 깊이 변화와 validation 성능 비교

plt.figure(figsize=(8, 5))
plt.plot(vgg_hist["val_avg_mf1"], label="VGG-16")
plt.plot(r18_hist["val_avg_mf1"], label="ResNet-18")
plt.plot(r50_hist["val_avg_mf1"], label="ResNet-50")
plt.xlabel("Epoch")
plt.ylabel("Validation Avg-Macro-F1")
plt.title("Validation Avg-MF1 by Backbone")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 분석 3) Loss 가중치 민감도

loss_weight_df = pd.DataFrame([
    best_summary("resnet18_w111", r18_hist),
    best_summary("resnet18_w211", r18_w211_hist),
    best_summary("resnet18_w121", r18_w121_hist),
    best_summary("resnet18_w112", r18_w112_hist),
])

loss_weight_df.insert(1, "loss_weights_weather/scene/timeofday", ["1/1/1", "2/1/1", "1/2/1", "1/1/2"])
loss_weight_df = add_hardest_attribute(loss_weight_df)

loss_weight_df